# AMB Training - Curriculum Learning

Progressive training: simple structures → complex structures

In [ ]:
!pip install nbtlib -q
print("[1] Dependencies ✓")

In [ ]:
import os, time, gzip, random
import numpy as np
from pathlib import Path
from typing import List, Optional, Dict

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

import nbtlib

DEVICE = torch.device('cuda')
print(f"CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0)}")
print("[2] Imports ✓")

In [ ]:
# ========== CONFIG ==========
DATASET = '/kaggle/input/mc-data'
MAX_SIZE = 16

# Curriculum stages: (max_structures, epochs, description)
CURRICULUM = [
    (20, 50, "Simple: 20 small structures"),
    (50, 50, "Medium: 50 structures"),
    (100, 50, "Complex: 100 structures"),
    (200, 50, "Full: 200 structures"),
]

BATCH = 64
LR = 3e-4
D_MODEL = 128
# ============================
print("[3] Config ✓")
for i, (n, e, desc) in enumerate(CURRICULUM):
    print(f"  Stage {i+1}: {desc} ({e} epochs)")

In [ ]:
# Roles - not converting bottom to FLOOR (fixed bias issue)
class Role:
    AIR, WALL, FLOOR, ROOF, WINDOW, DOOR = 0, 1, 2, 3, 4, 5

# Better mapping - keep walls as walls, only explicit floors as floors
BLOCK_MAP = {
    0: Role.AIR,
    # Glass = Window
    20: Role.WINDOW, 102: Role.WINDOW, 160: Role.WINDOW,
    # Doors
    64: Role.DOOR, 71: Role.DOOR, 193: Role.DOOR,
    # Stairs/Slabs = Roof
    44: Role.ROOF, 53: Role.ROOF, 67: Role.ROOF,
    108: Role.ROOF, 109: Role.ROOF, 114: Role.ROOF,
    126: Role.ROOF, 128: Role.ROOF,
}

print("[4] Roles ✓")

In [ ]:
def load_sch(path):
    try:
        nbt = nbtlib.load(path)
        root = nbt.get('Schematic', nbt.get('', nbt))
        w, h, d = int(root['Width']), int(root['Height']), int(root['Length'])
        blocks = np.array(root.get('Blocks', root.get('BlockData', [])), dtype=np.uint8)
        if len(blocks) != w*h*d: return None
        return {'w': w, 'h': h, 'd': d, 'blocks': blocks.reshape(h,d,w).transpose(2,0,1)}
    except:
        return None

def simplify(blocks, sz):
    """Convert blocks to roles - FIXED: don't make everything FLOOR"""
    out = np.zeros((sz,sz,sz), np.int64)
    w,h,d = min(blocks.shape[0],sz), min(blocks.shape[1],sz), min(blocks.shape[2],sz)
    b = blocks[:w,:h,:d]
    
    # Start with all non-air as WALL (not FLOOR!)
    r = np.where(b > 0, Role.WALL, Role.AIR).astype(np.int64)
    
    # Apply specific mappings
    for bid, role in BLOCK_MAP.items():
        r[b == bid] = role
    
    # Only bottom layer solid blocks become FLOOR (not all!)
    # Make foundation thinner - just the very bottom Y=0 edge blocks
    for x in range(w):
        for z in range(d):
            if r[x, 0, z] == Role.WALL:
                # Check if it's on the edge (potential foundation)
                if x == 0 or x == w-1 or z == 0 or z == d-1:
                    r[x, 0, z] = Role.FLOOR
    
    out[:w,:h,:d] = r
    return out

def get_complexity(blocks):
    """Calculate structure complexity for curriculum sorting"""
    non_air = np.count_nonzero(blocks)
    if non_air == 0: return 0
    
    # Height factor
    ys = np.where(blocks > 0)[1]
    height = ys.max() - ys.min() + 1 if len(ys) > 0 else 0
    
    # Variety factor (how many different roles)
    unique_roles = len(np.unique(blocks[blocks > 0]))
    
    return non_air * height * unique_roles

# Test
files = list(Path(DATASET).rglob('*.schematic'))[:5]
for f in files:
    sch = load_sch(str(f))
    if sch:
        roles = simplify(sch['blocks'], MAX_SIZE)
        counts = {r: int((roles==r).sum()) for r in range(6) if (roles==r).sum() > 0}
        print(f"  {f.name}: {counts}")

print("[5] Loader ✓")

In [ ]:
class CurriculumDataset(Dataset):
    def __init__(self, path, max_size, max_struct, sort_by_complexity=True):
        self.states = []
        self.phases = []
        self.progs = []
        self.targets = []
        
        # Load all schematics with complexity scores
        all_files = list(Path(path).rglob('*.schematic'))
        print(f"Found {len(all_files)} schematic files")
        
        # Load and score structures
        structures = []
        for f in all_files[:max_struct * 5]:  # Load extra for filtering
            sch = load_sch(str(f))
            if not sch: continue
            
            roles = simplify(sch['blocks'], max_size)
            n_blocks = np.count_nonzero(roles)
            if n_blocks < 20 or n_blocks > 400:  # Filter extremes
                continue
            
            complexity = get_complexity(roles)
            structures.append((roles, complexity, str(f)))
        
        print(f"Valid structures: {len(structures)}")
        
        # Sort by complexity (simple first for curriculum)
        if sort_by_complexity:
            structures.sort(key=lambda x: x[1])
        
        # Take the first max_struct
        structures = structures[:max_struct]
        print(f"Using {len(structures)} structures")
        
        # Generate samples
        for roles, complexity, fname in structures:
            positions = []
            for x in range(max_size):
                for y in range(max_size):
                    for z in range(max_size):
                        if roles[x,y,z] > 0:
                            positions.append((x, y, z, int(roles[x,y,z])))
            
            # Sort by Y (bottom-up), then X, Z
            positions.sort(key=lambda p: (p[1], p[0], p[2]))
            
            # Compute height for phase assignment
            ys = [p[1] for p in positions]
            min_y, max_y = min(ys), max(ys)
            height = max_y - min_y + 1
            
            state = np.zeros((max_size, max_size, max_size), dtype=np.int64)
            n = len(positions)
            
            for t, (x, y, z, block) in enumerate(positions):
                # Better phase assignment based on relative height
                rel_y = (y - min_y) / max(height - 1, 1)
                if rel_y < 0.2:
                    phase = 0  # FOUNDATION
                elif rel_y > 0.8:
                    phase = 2  # ROOF
                else:
                    phase = 1  # WALL
                
                # Override for specific roles
                if block == Role.WINDOW: phase = 3
                if block == Role.DOOR: phase = 4
                
                prog = t / max(n, 1)
                
                self.states.append(torch.from_numpy(state.copy()))
                self.phases.append(phase)
                self.progs.append(prog)
                self.targets.append((x, y, z, block))
                
                state[x, y, z] = block
            
            # STOP sample
            self.states.append(torch.from_numpy(state.copy()))
            self.phases.append(4)
            self.progs.append(1.0)
            self.targets.append((0, 0, 0, 0))
        
        # Print role distribution
        block_counts = {}
        for _, _, _, b in self.targets:
            block_counts[b] = block_counts.get(b, 0) + 1
        print(f"Role distribution: {block_counts}")
        print(f"Total samples: {len(self.states)}")
    
    def __len__(self): return len(self.states)
    
    def __getitem__(self, i):
        x, y, z, block = self.targets[i]
        return {
            'state': self.states[i],
            'phase': torch.tensor(self.phases[i], dtype=torch.long),
            'progress': torch.tensor(self.progs[i], dtype=torch.float32),
            'x': torch.tensor(x, dtype=torch.long),
            'y': torch.tensor(y, dtype=torch.long),
            'z': torch.tensor(z, dtype=torch.long),
            'block': torch.tensor(block, dtype=torch.long)
        }

print("[6] CurriculumDataset ✓")

In [ ]:
class SmallModel(nn.Module):
    def __init__(self, sz=16, d=128):
        super().__init__()
        self.sz = sz
        self.embed = nn.Embedding(6, 16)
        self.conv = nn.Sequential(
            nn.Conv3d(16, 32, 3, padding=1, stride=2),
            nn.ReLU(),
            nn.Conv3d(32, 64, 3, padding=1, stride=2),
            nn.ReLU(),
            nn.Conv3d(64, d, 3, padding=1, stride=2),
            nn.ReLU(),
            nn.AdaptiveAvgPool3d(1)
        )
        self.phase_emb = nn.Embedding(5, d//2)
        self.prog_fc = nn.Linear(1, d//2)
        self.fc = nn.Linear(d*2, d)
        self.pos_head = nn.Linear(d, sz**3)
        self.blk_head = nn.Linear(d, 6)
    
    def forward(self, state, phase, prog):
        x = self.embed(state).permute(0,4,1,2,3).float()
        x = self.conv(x).flatten(1)
        p = self.phase_emb(phase)
        g = self.prog_fc(prog.unsqueeze(-1))
        x = self.fc(torch.cat([x, p, g], -1))
        return self.pos_head(x), self.blk_head(x)

print("[7] Model ✓")

In [ ]:
def train_epoch(model, loader, opt, pos_fn, blk_fn, sz):
    model.train()
    tot_loss, tot_pacc, tot_bacc, n = 0, 0, 0, 0
    
    for batch in loader:
        state = batch['state'].to(DEVICE)
        phase = batch['phase'].to(DEVICE)
        prog = batch['progress'].to(DEVICE)
        tx, ty, tz = batch['x'].to(DEVICE), batch['y'].to(DEVICE), batch['z'].to(DEVICE)
        tblk = batch['block'].to(DEVICE)
        
        opt.zero_grad()
        pos_log, blk_log = model(state, phase, prog)
        
        tgt_idx = tx * sz**2 + ty * sz + tz
        loss = pos_fn(pos_log, tgt_idx) + blk_fn(blk_log, tblk)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        
        tot_loss += loss.item()
        tot_pacc += (pos_log.argmax(-1) == tgt_idx).float().mean().item()
        tot_bacc += (blk_log.argmax(-1) == tblk).float().mean().item()
        n += 1
    
    return tot_loss/n, tot_pacc/n, tot_bacc/n

print("[8] Training fn ✓")

In [ ]:
# Main curriculum training loop
model = SmallModel(MAX_SIZE, D_MODEL).to(DEVICE)
opt = AdamW(model.parameters(), lr=LR)

pos_fn = nn.CrossEntropyLoss()
blk_weights = torch.ones(6, device=DEVICE)
blk_weights[0] = 30.0  # STOP weight
blk_weights[2] = 0.5   # Lower FLOOR weight (less common now)
blk_fn = nn.CrossEntropyLoss(weight=blk_weights)

print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")
print("="*70)

best_loss = float('inf')
all_history = []

for stage_idx, (max_struct, epochs, desc) in enumerate(CURRICULUM):
    print(f"\n{'='*70}")
    print(f"STAGE {stage_idx+1}: {desc}")
    print(f"{'='*70}")
    
    # Load dataset for this stage
    ds = CurriculumDataset(DATASET, MAX_SIZE, max_struct, sort_by_complexity=True)
    
    if len(ds) == 0:
        print("No samples, skipping stage")
        continue
    
    loader = DataLoader(ds, BATCH, shuffle=True, num_workers=4, pin_memory=True)
    scheduler = CosineAnnealingLR(opt, epochs)
    
    print(f"Samples: {len(ds):,}, Batches: {len(loader)}")
    print("-"*70)
    
    for ep in range(epochs):
        t0 = time.time()
        loss, pacc, bacc = train_epoch(model, loader, opt, pos_fn, blk_fn, MAX_SIZE)
        scheduler.step()
        t = time.time() - t0
        
        all_history.append({'stage': stage_idx, 'epoch': ep, 'loss': loss, 'pos': pacc, 'blk': bacc})
        
        if (ep + 1) % 10 == 0 or ep == 0:
            print(f"Ep {ep+1:3d}/{epochs} | Loss {loss:.3f} | Pos {pacc:.3f} | Blk {bacc:.3f} | {t:.1f}s")
        
        if loss < best_loss:
            best_loss = loss
            torch.save(model.state_dict(), 'best.pt')
    
    # Save stage checkpoint
    torch.save(model.state_dict(), f'stage{stage_idx+1}.pt')
    print(f"Stage {stage_idx+1} complete. Best loss: {best_loss:.4f}")

print("\n" + "="*70)
print(f"[9] Curriculum training complete! Final best loss: {best_loss:.4f} ✓")

In [ ]:
# Test generation
model.eval()
state = torch.zeros(1, MAX_SIZE, MAX_SIZE, MAX_SIZE, dtype=torch.long, device=DEVICE)
phase = torch.tensor([0], device=DEVICE)

placed = 0
block_counts = {}

for step in range(500):
    prog = torch.tensor([step/500], dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        pos_log, blk_log = model(state, phase, prog)
    
    pos_idx = pos_log.argmax().item()
    blk = blk_log.argmax().item()
    
    if blk == 0:
        print(f"STOP at step {step}")
        break
    
    z = pos_idx % MAX_SIZE
    y = (pos_idx // MAX_SIZE) % MAX_SIZE
    x = pos_idx // (MAX_SIZE**2)
    
    if state[0,x,y,z] == 0:
        state[0,x,y,z] = blk
        placed += 1
        block_counts[blk] = block_counts.get(blk, 0) + 1
        
        # Update phase dynamically
        if y > 10: phase[0] = 2
        elif y > 2: phase[0] = 1

ROLE_NAMES = ['STOP', 'WALL', 'FLOOR', 'ROOF', 'WINDOW', 'DOOR']
print(f"\n[10] Generated: {placed} blocks ✓")
print("Block distribution:")
for b, c in sorted(block_counts.items()):
    print(f"  {ROLE_NAMES[b]}: {c}")